# Face Anonymization from 5 Given Landmarks

Pure-geometric face-removal pipeline for Einstar photogrammetry scans. Takes the
5 standard anatomical landmarks (Nz, Iz, Cz, Lpa, Rpa) as input and deletes the
facial region, then reverts the result back to the raw Einstar (digitized)
frame so the saved `.obj` + `.tsv` are a drop-in replacement for the original
scan at co-registration time.

**User-visible flow:** load scan &rarr; pick 5 landmarks &rarr; review the
anonymization mask &rarr; review the anonymized scan &rarr; save. All
intermediate transforms (normalize, align to CTF, build mask, delete verts)
run inside a single bundled cell.


In [53]:
import logging

import numpy as np
import pyvista as pv
import xarray as xr

import cedalion
import cedalion.dataclasses as cdc
import cedalion.io
import cedalion.vis.blocks
from cedalion.geometry.photogrammetry.anonymization import (
    normalize_axes,
    isolate_head,
    align_axes_from_landmarks,
    detect_cap_boundary,
    face_mask_from_landmarks,
    delete_masked_vertices,
    revert_to_einstar_frame,
    save_anonymized_scan,
)
from cedalion.vtktutils import trimesh_to_vtk_polydata
from cedalion.dataclasses import VTKSurface

logging.getLogger('cedalion').setLevel(logging.WARNING)
pv.set_jupyter_backend('server')

# === CONFIG ===
SUBJECT_NUMBER = 17
SCANS_FOLDER = '/home/ma7/BA/PG_Subjects'

# True: run the picker. False: use cached landmarks (fill after first pick).
INTERACTIVE = True

# Mask tunables (mm)
LANDMARK_KEEP_RADIUS = 8.0      # preserve a small sphere around each landmark
EAR_DELETE_RADIUS_MM = 40.0     # radius of the delete sphere around Lpa / Rpa

# Cap boundary detection tunables
CAP_BAND_WIDTH_MM = 15.0
CAP_BIN_SIZE_MM = 1.0
CAP_FOOT_GRAD_THRESHOLD = 0.2


## 1. Load the Einstar scan

In [54]:
path = f'{SCANS_FOLDER}/Subject{SUBJECT_NUMBER}/Subject{SUBJECT_NUMBER}.obj'
surface = cedalion.io.read_einstar_obj(path)
print(f'Loaded: {surface.nvertices:,} vertices, {surface.nfaces:,} faces')

# In trimesh 4.6 the texture image lives on visual.material.image, not
# visual.image. Verify it's there; if missing, fall back to the sibling JPG.
import os
import trimesh as _tm
from PIL import Image as _PILImage

def _visual_image(visual):
    img = getattr(visual, 'image', None)
    if img is None:
        mat = getattr(visual, 'material', None)
        img = getattr(mat, 'image', None) if mat is not None else None
    return img

_img = _visual_image(surface.mesh.visual)
if _img is None:
    _jpg = path.replace('.obj', '.jpg')
    _uv = getattr(surface.mesh.visual, 'uv', None)
    assert os.path.exists(_jpg) and _uv is not None, (
        f'no texture: jpg_exists={os.path.exists(_jpg)}, has_uv={_uv is not None}'
    )
    surface.mesh.visual = _tm.visual.TextureVisuals(
        uv=_uv, image=_PILImage.open(_jpg).convert('RGBA'),
    )
    _img = _visual_image(surface.mesh.visual)
    assert _img is not None, 'attach failed'
    print(f'Attached texture from JPG: {_img.size}')
else:
    print(f'Texture already attached: {_img.size}')

Loaded: 1,284,667 vertices, 2,223,716 faces
Texture already attached: (4016, 3776)


## 2. Pick the 5 landmarks interactively

Uses cedalion's built-in picker (`cedalion.vis.blocks.plot_surface(..., pick_landmarks=True)`,
authored by Mariia Iudina, upstream cedalion). Right-click on the mesh to place a
sphere; click again on the sphere to cycle its label through
`Nz -> Iz -> Cz -> Lpa -> Rpa`, or cycle past `Rpa` back to `Nz` to remove it.

Close the window when all 5 are placed with correct labels.

In [55]:
if INTERACTIVE:
    pvplt = pv.Plotter()
    get_landmarks = cedalion.vis.blocks.plot_surface(
        pvplt, surface, opacity=1.0, pick_landmarks=True
    )
    pvplt.show()

Widget(value='<iframe src="http://localhost:39989/index.html?ui=P_0x7f897374bc90_18&reconnect=auto" class="pyv…

## 3. Retrieve picked points and wrap into a `LabeledPointCloud`

Follows the pattern from `41_photogrammetric_optode_coregistration.ipynb` cells 26-28.

In [57]:
if INTERACTIVE:
    landmark_coordinates, landmark_labels = get_landmarks()
    landmark_coordinates = np.asarray(landmark_coordinates)
else:
    # Cached fallback: paste coordinates from a previous interactive run here.
    landmark_labels = ['Nz', 'Iz', 'Cz', 'Lpa', 'Rpa']
    landmark_coordinates = np.asarray([
        [0.0, 0.0, 0.0],   # Nz  -- replace
        [0.0, 0.0, 0.0],   # Iz  -- replace
        [0.0, 0.0, 0.0],   # Cz  -- replace
        [0.0, 0.0, 0.0],   # Lpa -- replace
        [0.0, 0.0, 0.0],   # Rpa -- replace
    ])

assert len(set(landmark_labels)) == 5, 'Need exactly 5 distinct landmarks'
assert set(landmark_labels) == {'Nz', 'Iz', 'Cz', 'Lpa', 'Rpa'}, (
    f'Unexpected labels: {landmark_labels}'
)

landmarks = xr.DataArray(
    np.vstack(landmark_coordinates),
    dims=['label', 'digitized'],
    coords={
        'label': ('label', list(landmark_labels)),
        'type': ('label', [cdc.PointType.LANDMARK] * 5),
        'group': ('label', ['L'] * 5),
    },
).pint.quantify('mm')

display(landmarks)

Magnitude,[[169.8175706956947 84.17791333176854 425.43686048545544] [32.061202102756184 203.5137101504909 436.13680432494687] [67.9290163798706 85.38361615977699 562.5009791587428] [18.535420787153996 67.46210371239562 393.38863930640076] [69.81661273869598 -12.304862754118403 480.1660378769869]]
Units,millimeter


## 4. Run the anonymization pipeline

All the geometric work happens in one block:

1. `normalize_axes` + `isolate_head` rotate into a canonical orientation and
   strip shoulders/body.
2. `align_axes_from_landmarks` maps the mesh + landmarks into the CTF frame
   (+X=anterior, +Y=left, +Z=up, origin at the Lpa/Rpa midpoint). The rotation
   `R` and affine `M_ctf` are kept in scope so we can invert them later.
3. `detect_cap_boundary` + `face_mask_from_landmarks` build the boolean
   deletion mask from the 5 landmarks. Landmark neighbourhoods and a midline
   nasion strip are preserved for downstream co-registration.
4. `delete_masked_vertices` drops every triangle touching a masked vertex.


In [58]:
# Normalize axes + isolate head.
lm_raw = landmarks.pint.dequantify().values
idx = {lbl: i for i, lbl in enumerate(landmarks['label'].values)}
Nz_raw = lm_raw[idx['Nz']]

surface_n, Nz_n, R = normalize_axes(surface, Nz_raw)
landmarks_n = landmarks.pint.dequantify().copy(data=lm_raw @ R.T).pint.quantify()
surface_n, _ = isolate_head(surface_n, Nz_n)

# Align to CTF frame.
surface_h, landmarks_n, M_ctf = align_axes_from_landmarks(surface_n, landmarks_n)
lm_n = landmarks_n.pint.dequantify().values
Nz, Iz, Cz, Lpa, Rpa = (lm_n[idx[lbl]] for lbl in ['Nz', 'Iz', 'Cz', 'Lpa', 'Rpa'])
ear_mid = 0.5 * (Lpa + Rpa)

# Build face-region mask.
verts = np.asarray(surface_h.mesh.vertices)
mid_y = 0.5 * (Lpa[1] + Rpa[1])
cap_z, *_ = detect_cap_boundary(
    verts, Nz, Cz, ear_mid, mid_y,
    band_width=CAP_BAND_WIDTH_MM,
    bin_size=CAP_BIN_SIZE_MM,
    foot_grad_threshold=CAP_FOOT_GRAD_THRESHOLD,
)
mask, _ = face_mask_from_landmarks(
    verts, Nz, Iz, Cz, Lpa, Rpa,
    cap_z=cap_z,
    ear_delete_radius=EAR_DELETE_RADIUS_MM,
)

# Preserve small spheres around each landmark + a midline nasion strip.
for lm in (Nz, Iz, Cz, Lpa, Rpa):
    near = np.linalg.norm(verts - lm, axis=1) < LANDMARK_KEEP_RADIUS
    mask[near] = False
nasion_strip = (
    (verts[:, 2] >= Nz[2]) &
    (verts[:, 2] < cap_z) &
    (np.abs(verts[:, 1] - Nz[1]) < LANDMARK_KEEP_RADIUS) &
    (verts[:, 0] > ear_mid[0])
)
mask[nasion_strip] = False

# Delete masked vertices.
surface_anon = delete_masked_vertices(surface_h, mask)

print(f'S{SUBJECT_NUMBER}: {surface_h.nvertices:,} -> {surface_anon.nvertices:,} verts '
      f'(-{surface_h.nvertices - surface_anon.nvertices:,}, cap_z={cap_z:.1f} mm)')


S17: 678,081 -> 578,478 verts (-99,603, cap_z=10.0 mm)


## 5. Anonymization mask overlay

Red = will be deleted, white = kept. The 5 landmark spheres are overlaid for
orientation. If the red region hugs the face without leaking onto ears, top of
head, or neck, the geometric rules are working.


In [59]:
lm_colors = {'Nz': 'lime', 'Iz': 'magenta', 'Cz': 'cyan', 'Lpa': 'orange', 'Rpa': 'blue'}

pvplt = pv.Plotter()

vtk_surface = VTKSurface.from_trimeshsurface(surface_h)
pv_mesh = pv.wrap(vtk_surface.mesh)
pv_mesh['mask'] = mask.astype(float)
pvplt.add_mesh(
    pv_mesh, scalars='mask', cmap=['white', 'red'], clim=[0, 1],
    show_scalar_bar=False, opacity=1.0, smooth_shading=True,
)

for lbl, pos in zip(landmarks_n['label'].values, lm_n):
    c = lm_colors.get(lbl, 'yellow')
    pvplt.add_mesh(pv.Sphere(radius=4, center=pos), color=c)
    pvplt.add_point_labels(
        [pos], [lbl], font_size=16, text_color=c, shape=None, always_visible=True,
    )

pvplt.add_text(
    f'S{SUBJECT_NUMBER} | mask: {mask.sum():,} / {len(mask):,} verts '
    f'({100*mask.sum()/len(mask):.1f}%)',
    position='upper_left', font_size=12,
)
pvplt.show()

Widget(value='<iframe src="http://localhost:39989/index.html?ui=P_0x7f89737b6f10_19&reconnect=auto" class="pyv…

## 6. Revert to the Einstar frame and compare

Inverts the `normalize_axes` and `align_axes_from_landmarks` transforms so the
anonymized mesh + landmarks end up back in the raw Einstar (digitized) frame.
Both views below share the same coordinate system, so the comparison is
geometrically direct. This is also the frame the saved files will be in,
matching the co-registration tutorial's `read_einstar_obj` convention.


In [60]:
# Invert the two transforms so both mesh and landmarks are back in 'digitized'.
surface_anon_dig, landmarks_dig = revert_to_einstar_frame(
    surface_anon, landmarks_n, R, M_ctf,
)
lm_dig = landmarks_dig.pint.dequantify().values

orig_vtk = trimesh_to_vtk_polydata(surface.mesh)
anon_vtk = trimesh_to_vtk_polydata(surface_anon_dig.mesh)

lm_colors = {'Nz': 'lime', 'Iz': 'magenta', 'Cz': 'cyan', 'Lpa': 'orange', 'Rpa': 'blue'}
n_removed = surface_h.nvertices - surface_anon.nvertices

pvplt = pv.Plotter(shape=(1, 2))

pvplt.subplot(0, 0)
pvplt.add_mesh(pv.wrap(orig_vtk), rgb=True, smooth_shading=True)
for lbl, pos in zip(landmarks_dig['label'].values, lm_dig):
    pvplt.add_mesh(pv.Sphere(radius=4, center=pos),
                   color=lm_colors.get(lbl, 'yellow'))
pvplt.add_text(f'S{SUBJECT_NUMBER} Original', position='upper_left', font_size=14)

pvplt.subplot(0, 1)
pvplt.add_mesh(pv.wrap(anon_vtk), rgb=True, smooth_shading=True)
for lbl, pos in zip(landmarks_dig['label'].values, lm_dig):
    pvplt.add_mesh(pv.Sphere(radius=4, center=pos),
                   color=lm_colors.get(lbl, 'yellow'))
pvplt.add_text(
    f'S{SUBJECT_NUMBER} Anonymized (-{n_removed:,} verts)',
    position='upper_left', font_size=14,
)

pvplt.link_views()
pvplt.show()


Widget(value='<iframe src="http://localhost:39989/index.html?ui=P_0x7f89737d1a10_20&reconnect=auto" class="pyv…

## 7. Save

Writes the anonymized `.obj` + `.mtl` + sanitized `.jpg` bundle plus a
sidecar `_landmarks.tsv` (the format consumed at step 5.2 of the cedalion
photogrammetry co-registration tutorial). All carry `crs=digitized`. The
JPG is rebuilt from the original texture keeping only pixels still
referenced by the anonymized mesh's UVs; face-region pixels are blanked,
so opening the JPG alone reveals no face.


In [36]:
out = f'{SCANS_FOLDER}/Subject{SUBJECT_NUMBER}/Subject{SUBJECT_NUMBER}_anon.obj'
written = save_anonymized_scan(surface_anon_dig, out, landmarks=landmarks_dig)
print('Wrote:')
for p in written:
    print(f'  {p}')


Wrote:
  /home/ma7/BA/PG_Subjects/Subject15/Subject15_anon.jpg
  /home/ma7/BA/PG_Subjects/Subject15/Subject15_anon.mtl
  /home/ma7/BA/PG_Subjects/Subject15/Subject15_anon.obj
  /home/ma7/BA/PG_Subjects/Subject15/Subject15_anon_landmarks.tsv
